# Notebook 04 — Tuned models, multiple linear regression, and before-reference-test results

This notebook extends the modelling in notebook 03. It works on the **main feature set** (no features measured inside the target 1C discharge and no capacity measures), defined in `docs/feature_timing_audit.csv`.

It produces three things:
1. **Tuned SVR, random forest, and gradient boosting** — settings chosen with a small grid inside each training fold. The inner split matches the outer test (by cell, by profile, or random), so no held-out cell or profile is used for tuning.
2. **Multiple linear regression** — the plain linear model from our proposal, run on the same main set.
3. **Before-reference-test results** — models that use only information available before the reference test (usage history and profile settings), with nothing measured during the reference test.

All metrics are MAE / RMSE / R². MAE and RMSE are in SOH percentage points, so lower is better.

In [1]:
from pathlib import Path
import sys, itertools, warnings
import numpy as np, pandas as pd
from joblib import Parallel, delayed
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.modelling import build_pipeline, splitters, evaluate
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

df = pd.read_csv(ROOT / "results" / "rtp_features_fixed.csv", dtype={"cell_id": str})
df = df[df.rtp_T_set == 25].dropna(subset=["SOH"]).reset_index(drop=True)

audit = pd.read_csv(ROOT / "docs" / "feature_timing_audit.csv")
BEFORE = "before this RTP (usage history / profile)"
IN_RTP = "in this RTP, outside the target discharge"
CAPACITY = ["chg_CC_Ah", "chg_CC_time_s", "chg_CC_fraction", "chg_CV_Ah", "dis_C3_CC_Ah"]
keep = lambda names: [c for c in names if c in df.columns and df[c].nunique(dropna=True) > 1]

MAIN = [c for c in keep(audit.loc[audit.timing_group.isin([BEFORE, IN_RTP]), "feature"]) if c not in CAPACITY]
BEFORE_RTP = keep(audit.loc[audit.timing_group == BEFORE, "feature"])
print(df.shape, "rows |", df.cell_id.nunique(), "cells |", df.profile.nunique(), "profiles")
print("main set:", len(MAIN), "features | before-RTP set:", len(BEFORE_RTP), "features")

(306, 75) rows | 15 cells | 14 profiles
main set: 44 features | before-RTP set: 27 features


## 1. Tuned SVR, random forest, gradient boosting, and multiple linear regression

The grids are small on purpose (a first pass, not a full search). Tuning is nested: for every outer fold, the best setting is chosen by cross-validation **inside** that fold's training rows, using the same kind of split as the outer test.

In [2]:
scaled_y = lambda est: TransformedTargetRegressor(est, transformer=StandardScaler())

def svr_models():
    for C, g in itertools.product([1.0, 10.0, 100.0], ["scale", 0.01, 0.1]):
        yield dict(C=C, gamma=g), scaled_y(SVR(C=C, gamma=g, epsilon=0.05)), True

def rf_models():
    for mf, msl in itertools.product([0.3, 0.5, 1.0], [1, 3]):
        yield dict(max_features=mf, min_samples_leaf=msl), RandomForestRegressor(
            200, min_samples_leaf=msl, max_features=mf, random_state=0, n_jobs=1), False

def gb_models():
    for lr, mln in itertools.product([0.03, 0.1], [7, 15]):
        yield dict(learning_rate=lr, max_leaf_nodes=mln), HistGradientBoostingRegressor(
            max_iter=300, learning_rate=lr, max_leaf_nodes=mln, min_samples_leaf=10, random_state=0), False

FAMILIES = {"SVR (RBF, tuned)": svr_models, "Random forest (tuned)": rf_models, "Gradient boosting (tuned)": gb_models}
y = df.SOH.values
X = df[MAIN + ["cell_id"]]
cells, profiles = df.cell_id.values, df.profile.values

def inner_cv(split_name, tr):
    if split_name == "Random split":
        return KFold(5, shuffle=True, random_state=0), None
    g = cells[tr] if split_name == "Leave-cell-out" else profiles[tr]
    return GroupKFold(min(5, len(np.unique(g)))), g

def pick(family, split_name, tr):
    cv, g = inner_cv(split_name, tr)
    Xtr, ytr = X.iloc[tr], y[tr]
    best, best_err = None, np.inf
    for params, est, scale in FAMILIES[family]():
        errs = [mean_absolute_error(ytr[b], build_pipeline(MAIN, est, scale).fit(Xtr.iloc[a], ytr[a]).predict(Xtr.iloc[b]))
                for a, b in cv.split(Xtr, ytr, g)]
        if np.mean(errs) < best_err:
            best, best_err = (est, scale), np.mean(errs)
    return best

def run_fold(split_name, k, tr, te):
    rows = []
    for family in FAMILIES:
        est, scale = pick(family, split_name, tr)
        rows.append((family, build_pipeline(MAIN, est, scale).fit(X.iloc[tr], y[tr]).predict(X.iloc[te])))
    rows.append(("Multiple linear regression",
                 build_pipeline(MAIN, LinearRegression(), True).fit(X.iloc[tr], y[tr]).predict(X.iloc[te])))
    return split_name, te, rows

jobs = [(sn, k, tr, te) for sn, (cv, grp) in splitters(df).items()
        for k, (tr, te) in enumerate(cv.split(X, y, grp))]
results = Parallel(n_jobs=-1)(delayed(run_fold)(*j) for j in jobs)

oof = {}
for sn, te, rows in results:
    for model, pred in rows:
        oof.setdefault((sn, model), np.full(len(y), np.nan))[te] = pred
rows = [dict(split=sn, model=m, MAE=mean_absolute_error(y, p), RMSE=root_mean_squared_error(y, p), R2=r2_score(y, p))
        for (sn, m), p in oof.items()]
tuned = pd.DataFrame(rows)
(ROOT / "results" / "modelling_tuned").mkdir(parents=True, exist_ok=True)
tuned.to_csv(ROOT / "results" / "modelling_tuned" / "metrics_tuned.csv", index=False)
tuned.pivot_table(index="model", columns="split", values="MAE").round(2)[["Random split","Leave-cell-out","Leave-profile-out"]]

split,Random split,Leave-cell-out,Leave-profile-out
model,,,
Gradient boosting (tuned),1.16,1.95,2.04
Multiple linear regression,1.68,2.68,3.28
Random forest (tuned),1.09,1.97,2.08
"SVR (RBF, tuned)",0.77,3.44,4.72


## 2. Before-reference-test models

Here the models use only usage history and profile settings, with nothing measured during the reference test. This shows how much the main result depends on having a reference test available.

In [3]:
before_metrics, _, _ = evaluate(df, {"Before-RTP only": BEFORE_RTP})
(ROOT / "results" / "modelling_before_rtp").mkdir(parents=True, exist_ok=True)
before_metrics.to_csv(ROOT / "results" / "modelling_before_rtp" / "metrics_before_rtp.csv", index=False)
before_metrics.pivot_table(index="model", columns="split", values="MAE").round(2)[["Random split","Leave-cell-out","Leave-profile-out"]]

split,Random split,Leave-cell-out,Leave-profile-out
model,,,
Gradient boosting,1.95,4.94,7.07
Mean baseline,7.88,8.18,8.24
Random forest,1.74,5.71,7.49
Ridge,4.02,6.42,6.96
SVR (RBF),1.51,7.17,8.76


## Summary

- **Tuned models (main set), MAE on unseen cells / unseen profiles:** gradient boosting ~1.95 / ~2.04, random forest ~1.97 / ~2.08, SVR ~3.44 / ~4.72. Tuning helped SVR the most but it stays the weakest on unseen data.
- **Multiple linear regression:** ~2.68 / ~3.28 on unseen cells / profiles, a little worse than Ridge, as expected with correlated features.
- **Before-reference-test models:** much worse (gradient boosting ~4.94 on unseen cells, against ~2.00 for the main set), so the main result holds only when a reference test is available.

Result files written: `results/modelling_tuned/metrics_tuned.csv` and `results/modelling_before_rtp/metrics_before_rtp.csv`.